# Thesis 2024 - 2025
## UGBS
### Isil Oral

* Project on green gentrification in Vienna through social status index, green spaces, blue areas, and housing prices

## Data on Airbnb Listings in Vienna
* Gabor Bekes' data
* obtained from https://insideairbnb.com/get-the-data/

In [1]:
from shapely.geometry import box, Point, LineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import geopandas as gpd
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling
import osmnx as ox  
import contextily as ctx
from geopy.geocoders import Nominatim  
from geopy.extra.rate_limiter import RateLimiter
import os
# !pip install h3
import h3
import matplotlib.patches as mpatches
import seaborn as sns
#!pip install scipy
from scipy.stats import skew
import statsmodels.api as sm

In [2]:
# Get the current working directory
current_path = os.getcwd()

# Define relative paths for data, output, and helper functions
data_in = os.path.join(current_path, "data")       # Folder with input data
output = os.path.join(current_path, "output")      # Folder for saving results
df_dir = os.path.join(current_path, "dataframes")

In [3]:
# Load data sets
gdf_census = gpd.read_file('data/data_ssi.geojson')
census_boundaries = gpd.read_file('data/ZAEHLBEZIRKOGDPolygon.shp')
airbnb = pd.read_csv('data/airbnb_listings.csv')

In [4]:
# Load any dataframe from that folder
admin_city = gpd.read_file(os.path.join(df_dir, "admin_city_vienna.geojson"))
urban_green_spaces =gpd.read_file(os.path.join(df_dir, "urban_green_spaces.geojson"))
urban_blue_spaces = gpd.read_file(os.path.join(df_dir, "urban_blue_spaces.geojson"))
ugbs_socioecon_census = gpd.read_file(os.path.join(df_dir, "ugbs_socioecon_census.geojson"))
socioecon_districts = gpd.read_file(os.path.join(df_dir, "socioecon_districts.geojson"))
housingprices_district = gpd.read_file(os.path.join(df_dir, "housingprices_district.geojson"))
social_housing = pd.read_csv(os.path.join(df_dir, "social_housing.csv"))
social_housing_gdf = gpd.read_file(os.path.join(df_dir, "social_housing.geojson"))
aptfeatures_social_census = pd.read_csv(os.path.join(df_dir, "aptfeatures_social_census.csv"))
apt_features_final_census = pd.read_csv(os.path.join(df_dir, "apt_features_final_census.csv"))

Skipping field ways: unsupported OGR type: 1


### Initial Examination

In [5]:
airbnb.columns.to_list()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

In [6]:
airbnb_listings = airbnb[['id','neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                         'latitude', 'longitude',
                         'property_type', 'room_type', 'bathrooms',
                         'bedrooms',
                         'amenities',
                         'price',
                         'review_scores_rating', 'review_scores_accuracy', 'number_of_reviews']].copy()

print(len(airbnb))

# Remove $ sign and any commas, then convert to float
airbnb_listings['price'] = (
    airbnb_listings['price']
    .astype(str)  # in case some values are not strings
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Ensure numeric conversion
airbnb_listings['bathrooms'] = pd.to_numeric(airbnb_listings['bathrooms'], errors='coerce')
airbnb_listings['bedrooms'] = pd.to_numeric(airbnb_listings['bedrooms'], errors='coerce')
airbnb_listings['price'] = pd.to_numeric(airbnb_listings['price'], errors='coerce')

airbnb_stats = airbnb_listings.groupby('property_type').agg(
    bathrooms_min=('bathrooms', 'min'),
    bathrooms_mean=('bathrooms', 'mean'),
    bathrooms_max=('bathrooms', 'max'),
    bathrooms_median=('bathrooms', 'median'),
    
    bedrooms_min=('bedrooms', 'min'),
    bedrooms_mean=('bedrooms', 'mean'),
    bedrooms_max=('bedrooms', 'max'),
    bedrooms_median=('bedrooms', 'median'),
    
    price_min=('price', 'min'),
    price_mean=('price', 'mean'),
    price_max=('price', 'max'),
    price_median=('price', 'median'),
    
    count=('price', 'count')
).round(2).reset_index()

print(len(airbnb_stats))
airbnb_stats

13790
48


,property_type,bathrooms_min,bathrooms_mean,bathrooms_max,bathrooms_median,bedrooms_min,bedrooms_mean,bedrooms_max,bedrooms_median,price_min,price_mean,price_max,price_median,count
0,Barn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Casa particular,1.0,1.00,1.0,1.00,1.0,1.00,1.0,1.0,51.0,51.00,51.0,51.0,1
2,Entire bungalow,1.0,1.62,3.0,1.25,1.0,1.50,3.0,1.0,80.0,137.00,181.0,143.5,4
3,Entire chalet,1.0,1.00,1.0,1.00,2.0,2.00,2.0,2.0,120.0,120.00,120.0,120.0,1
4,Entire condo,1.0,1.22,5.5,1.00,0.0,1.28,7.0,1.0,24.0,107.97,936.0,82.0,584
5,Entire cottage,0.5,1.00,1.5,1.00,0.0,1.25,2.0,1.5,42.0,95.50,129.0,105.5,4
6,Entire guest suite,1.0,1.17,1.5,1.00,1.0,1.67,3.0,1.5,98.0,110.67,125.0,109.0,3
7,Entire guesthouse,1.0,1.00,1.0,1.00,1.0,1.06,2.0,1.0,55.0,86.25,128.0,86.0,16
8,Entire home,1.0,1.68,6.5,1.50,1.0,2.53,11.0,2.0,38.0,192.27,832.0,147.0,90
9,Entire loft,1.0,1.22,3.0,1.00,0.0,1.16,4.0,1.0,41.0,118.75,720.0,96.0,87


In [7]:
# Filter rows where 'property_type' contains 'Entire'
airbnb_entireunits = airbnb_listings[airbnb_listings['property_type'].str.contains('Entire', case=False, na=False)].copy()

entiretype_counts = airbnb_entireunits['property_type'].value_counts()
print(entiretype_counts)

# Group units to later on create proper dummies for similar units
# Define grouping dictionary
group_map = {
    'Entire rental unit': 'Entire Apartment',
    'Entire condo': 'Entire Apartment',
    'Entire serviced apartment': 'Entire Apartment',
    'Entire loft': 'Entire Apartment',
    
    'Entire home': 'Entire House',
    'Entire guesthouse': 'Entire House',
    'Entire vacation home': 'Entire House',
    'Entire villa': 'Entire House',
    'Entire townhouse': 'Entire House',
    'Entire guest suite': 'Entire House',
    'Entire bungalow': 'Entire House',
    'Entire cottage': 'Entire House',
    'Entire chalet': 'Entire House',

    'Entire place': 'Other'
}

# Apply mapping
airbnb_entireunits['property_type_grouped'] = airbnb_entireunits['property_type'].map(group_map).fillna('Other')

# Optionally check value counts
print(airbnb_entireunits['property_type_grouped'].value_counts())

# Create geometry column from lat/lon
airbnb_entireunits['geometry'] = airbnb_entireunits.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Convert to GeoDataFrame
airbnb_entireunits_gdf = gpd.GeoDataFrame(airbnb_entireunits, geometry='geometry', crs='EPSG:4326')  # assuming WGS84


property_type
Entire rental unit           9602
Entire condo                  774
Entire serviced apartment     442
Entire loft                   125
Entire home                   118
Entire vacation home           25
Entire guesthouse              17
Entire villa                   14
Entire place                   10
Entire townhouse                9
Entire guest suite              7
Entire bungalow                 6
Entire cottage                  4
Entire chalet                   1
Name: count, dtype: int64
property_type_grouped
Entire Apartment    10943
Entire House          201
Other                  10
Name: count, dtype: int64


In [8]:
airbnb_entireunits_gdf.head(3)

,id,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,bathrooms,bedrooms,amenities,price,review_scores_rating,review_scores_accuracy,number_of_reviews,property_type_grouped,geometry
0,38768,Leopoldstadt,NaN,48.21924,16.37831,Entire rental unit,Entire home/apt,1.0,1.0,"[""Shared patio or balcony"", ""Courtyard view"", ...",80.0,4.75,4.81,398,Entire Apartment,POINT (16.37831 48.21924)
1,40625,Rudolfsheim-Fnfhaus,NaN,48.18434,16.32701,Entire rental unit,Entire home/apt,1.0,2.0,"[""Drying rack for clothing"", ""Freezer"", ""Stain...",70.0,4.86,4.91,221,Entire Apartment,POINT (16.32701 48.18434)
2,51287,Leopoldstadt,NaN,48.21778,16.37847,Entire rental unit,Entire home/apt,1.0,0.0,"[""Host greets you"", ""Iron"", ""Bed linens"", ""Smo...",61.0,4.67,4.78,381,Entire Apartment,POINT (16.37847 48.21778)


In [9]:
airbnb_entireunits_gdf.to_file("dataframes/airbnb_entireunits.geojson", driver="GeoJSON")

### Aggregating 9 month airbnb data

In [10]:
airbnb_dec12 = pd.read_csv('data/listings_20241212.csv')
airbnb_sept11 = pd.read_csv('data/listings_20240911.csv')
airbnb_june15 = pd.read_csv('data/listings_20240615.csv')

In [11]:
airbnb_dec12 = airbnb_dec12[['id','neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                         'latitude', 'longitude',
                         'property_type', 'room_type', 'bathrooms',
                         'bedrooms',
                         'amenities',
                         'price',
                         'review_scores_rating', 'review_scores_accuracy', 'number_of_reviews']].copy()

airbnb_sept11 = airbnb_sept11[['id','neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                         'latitude', 'longitude',
                         'property_type', 'room_type', 'bathrooms',
                         'bedrooms',
                         'amenities',
                         'price',
                         'review_scores_rating', 'review_scores_accuracy', 'number_of_reviews']].copy()

airbnb_june15 = airbnb_june15[['id','neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                         'latitude', 'longitude',
                         'property_type', 'room_type', 'bathrooms',
                         'bedrooms',
                         'amenities',
                         'price',
                         'review_scores_rating', 'review_scores_accuracy', 'number_of_reviews']].copy()

In [12]:
# Keep only id, price and review; rename price and review based on years
dec12 = airbnb_dec12[['id', 'price', 'review_scores_rating']].copy()
dec12.rename(columns={'price': 'price_20241212',
                       'review_scores_rating': 'review_20241212'}, inplace=True)

sept11 = airbnb_sept11[['id', 'price', 'review_scores_rating']].copy()
sept11.rename(columns={'price': 'price_20240911',
                      'review_scores_rating': 'review_20240911'}, inplace=True)

june15 = airbnb_june15[['id', 'price', 'review_scores_rating']].copy()
june15.rename(columns={'price': 'price_20240615',
                      'review_scores_rating': 'review_20240615'}, inplace=True)

# airbnb_listings assumed to be base DataFrame with 'price' column (e.g., from your original scrape)
base = airbnb_entireunits_gdf[['id', 'price', 'review_scores_rating']].copy()
base.rename(columns={'price': 'price_20250305',
                    'review_scores_rating': 'review_20250305'}, inplace=True)

# Now merge cleanly
merged = base.merge(dec12, on='id', how='inner')
merged = merged.merge(sept11, on='id', how='inner')
merged = merged.merge(june15, on='id', how='inner')
merged.head(3)

,id,price_20250305,review_20250305,price_20241212,review_20241212,price_20240911,review_20240911,price_20240615,review_20240615
0,38768,80.0,4.75,$81.00,4.74,$100.00,4.74,$89.00,4.74
1,40625,70.0,4.86,$135.00,4.86,$169.00,4.86,$169.00,4.85
2,51287,61.0,4.67,$86.00,4.67,$80.00,4.67,$100.00,4.67


In [13]:
initial_ids = set(airbnb_listings['id'])
final_ids = set(merged['id'])

print(f"Number of IDs removed (not found in all datasets): {len(initial_ids - final_ids)}")
print(f"Listings retained in all snapshots: {len(merged)}")

Number of IDs removed (not found in all datasets): 5351
Listings retained in all snapshots: 8439


In [14]:
price_columns = ['price_20250305', 'price_20240615', 'price_20240911', 'price_20241212']
review_columns = ['review_20250305', 'review_20240615', 'review_20240911', 'review_20241212']

# Clean and convert each to float
for col in price_columns:
    merged[col] = (
        merged[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

for col in review_columns:
    merged[col] = (
        merged[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )


merged['avg_price'] = merged[price_columns].mean(axis=1)
merged['avg_review'] = merged[review_columns].mean(axis=1)
merged.head(3)

,id,price_20250305,review_20250305,price_20241212,review_20241212,price_20240911,review_20240911,price_20240615,review_20240615,avg_price,avg_review
0,38768,80.0,4.75,81.0,4.74,100.0,4.74,89.0,4.74,87.50,4.7425
1,40625,70.0,4.86,135.0,4.86,169.0,4.86,169.0,4.85,135.75,4.8575
2,51287,61.0,4.67,86.0,4.67,80.0,4.67,100.0,4.67,81.75,4.6700


In [15]:
# Select columns you want to add
cols_to_add = ['id', 'neighbourhood_cleansed',
               'latitude', 'longitude', 'property_type_grouped', 'property_type', 'bathrooms', 'bedrooms', 'amenities']

# Subset from base
airbnb_metadata = airbnb_entireunits_gdf[cols_to_add].copy()

# Merge back into the final DataFrame
annual_airbnb = merged.merge(airbnb_metadata, on='id', how='left')

In [16]:
annual_airbnb.head(3)

,id,price_20250305,review_20250305,price_20241212,review_20241212,price_20240911,review_20240911,price_20240615,review_20240615,avg_price,avg_review,neighbourhood_cleansed,latitude,longitude,property_type_grouped,property_type,bathrooms,bedrooms,amenities
0,38768,80.0,4.75,81.0,4.74,100.0,4.74,89.0,4.74,87.50,4.7425,Leopoldstadt,48.21924,16.37831,Entire Apartment,Entire rental unit,1.0,1.0,"[""Shared patio or balcony"", ""Courtyard view"", ..."
1,40625,70.0,4.86,135.0,4.86,169.0,4.86,169.0,4.85,135.75,4.8575,Rudolfsheim-Fnfhaus,48.18434,16.32701,Entire Apartment,Entire rental unit,1.0,2.0,"[""Drying rack for clothing"", ""Freezer"", ""Stain..."
2,51287,61.0,4.67,86.0,4.67,80.0,4.67,100.0,4.67,81.75,4.6700,Leopoldstadt,48.21778,16.37847,Entire Apartment,Entire rental unit,1.0,0.0,"[""Host greets you"", ""Iron"", ""Bed linens"", ""Smo..."


In [17]:
# Filter rows where 'property_type' contains 'Entire'
annual_airbnb = annual_airbnb[annual_airbnb['property_type'].str.contains('Entire', case=False, na=False)].copy()

# Create geometry column from lat/lon
annual_airbnb['geometry'] = annual_airbnb.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Convert to GeoDataFrame
annual_airbnb_gdf = gpd.GeoDataFrame(annual_airbnb, geometry='geometry', crs='EPSG:4326')  # assuming WGS84
annual_airbnb_gdf.to_file("dataframes/annual_airbnb_entire.geojson", driver="GeoJSON")